In [13]:
# Classifier lives in app.ml (service module). Visualization deps stay in the notebook.
from app.config import get_settings
from app.ml.arabic_classifier import AdvancedArabicClassifier
from app.ml.paths import resolve_arabert_model_path

# Optional: notebook-only evaluation helper (uses pandas)
import pandas as pd
from tqdm import tqdm


def predict_dataframe(classifier, df, threshold=None):
    df = df.fillna("")
    y_true, y_pred = [], []
    actual_threshold = (
        threshold if threshold is not None else classifier.default_threshold
    )
    for _, row in tqdm(df.iterrows(), total=len(df)):
        probs = classifier.predict_full_article(
            row.get("title", ""),
            row.get("keywords", ""),
            row.get("abstract", ""),
        )
        top_label = next(iter(probs))
        top_conf = next(iter(probs.values())) / 100.0
        pred_id = classifier.label2id[top_label]
        final_pred_id = (
            classifier.unspecified_id
            if (top_conf < actual_threshold and classifier.unspecified_id != -1)
            else pred_id
        )
        if "label" in row:
            y_true.append(int(row["label"]))
        y_pred.append(final_pred_id)
    return y_true, y_pred



In [ ]:
            """
            {
        "total_articles_analyzed": int,          # إجمالي عدد الصفحات/المقالات التي تم تحليلها (مثلاً: 32)
        "applied_threshold": float,              # عتبة الثقة المستخدمة في هذا الفحص (مثلاً: 0.0)
        
        "average_confidence_per_class": {       # قاموس يحتوي على متوسط الثقة لكل فئة مرتبة من الأعلى للأقل
            "اسم الفئة الأولى": float,              # (مثلاً: "العلوم القانونية": 85.42)
            "اسم الفئة الثانية": float,             # (مثلاً: "العلوم الاقتصادية": 12.15)
            # ... باقي الفئات الـ 10
        },
        
        "overall_dominant_class": str,           # نص يحتوي على اسم الفئة المسيطرة على البحث بالكامل (مثلاً: "العلوم القانونية")
        
        "individual_results": [                  # مصفوفة (List) تحتوي على تفاصيل كل صفحة/مقال بشكل منفرد
            {
                "final_prediction": str,         # التصنيف النهائي بعد تطبيق العتبة (إما اسم الفئة أو "غير محدد")
                "original_top": str,             # التصنيف الأصلي الأعلى قبل العتبة
                "confidence": float,             # نسبة ثقة الموديل في هذا التصنيف (مثلاً: 92.5)
                "all_probabilities": {           # قاموس التوزيع الاحتمالي الكامل لهذه الصفحة مرتبة تنازلياً
                    "الفئة الأعلى": float,
                    "الفئة التالية": float,
                    # ...
                }
            },
            # ... يتكرر هذا القاموس لكل صفحة داخل المصفوفة
        ]
    }
    """

In [2]:
articles_batch = [
    {
        "title": "الصفحة رقم 1",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nنظرية التفويض الإداري وإشكالاتها التطبيقية\nالملخص\nد. عمار التركاوي\nيؤدي التركيز الشديد للسلطة الإدارية إلى ارتباك في الجهاز الإداري، ومن ثم عجزه عن القيام بالمهمات المسندة إليه، ويعد التفويض الإداري من الوسائل الإدارية المهمة القيام المنظمة الإدارية بأعمالها على الصورة الأمثل، والكفاءة الأعلى، وهو أسلوب يحقق ديمقراطية الإدارة عن طريق إشراك المرؤوسين في صنع القرارات الإدارية.\nويهدف هذا البحث إلى التعريف بنظرية التفويض الإداري في محاورها وأبعادها المختلفة، من خلال دراسة مفهوم التفويض الإداري، وأنواعه، وأهم المبادئ التي يقوم عليها، والشروط القانونية اللازمة لصحته، وبيان آثاره، وتمييزه عن بعض المفاهيم والنظم القانونية المشابهة له.\nوتبين الدراسة أيضاً مزايا التفويض الإداري، وأهم الصعوبات التي يواجهها في الواقع العملي، وكيفية مواجهة هذه الصعوبات، سواء كانت تعود للرؤساء الإداريين أم للمرؤوسين، وصولاً إلى تحقيق الأداء الأفضل للمنظمة الإدارية.\nأستاذ مساعد في قسم القانون العام - كلية الحقوق - جامعة دمشق.\n165 [cite: 1, 2, 3, 4, 5, 6, 7, 8, 9]"
    },
    {
        "title": "الصفحة رقم 2",
        "abstract": "د. عمار التركاوي\nنظرية التفويض الإداري وإشكالاتها التطبيقية\nThe theory of administrative delegation\nAnd its technical problems\nDr. AmmarTerkawi\nAbstract\nExtreme concentration of authority leads to confusion in administrative organization and subsequently its inability to perform the assigned tasks.\nAdministrative delegation is considered as an important way to help the Administrative organization in doing its work optimally and efficiently.\nDelegation is a method that achieves democratic administration by involving subordinates in the process of making administrative decisions.\nThis study aims to define the administrative delegation theory, its dimensions and issues.\nBy studying administrative delegation concept, kinds, the most important underlying principles, the necessary legal requirements needed for administrative delegation validity, indication to the impact of the administrative delegation and distinguish it from other similar concepts and legal methods.\nThe study also indicates the administrative delegation advantages and the most important difficulties faced in practice and how to face these difficulties whether they happened because of subordinates or leaders in order to achieve the best performance of the administrative organization.\nAssociate professor in the department of public law, Faculty of Law - Damascus University.\n166 [cite: 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]"
    },
    {
        "title": "الصفحة رقم 3",
        "abstract": "المقدمة\nمجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nمن المتفق عليه أن توزيع الاختصاصات الإدارية في الدولة إما أن يقوم على أساس لامركزي، وإما على أساس مركزي.\nفإذا ما اتبعت الدولة الأسلوب اللامركزي في توزيع الاختصاصات الإدارية، فمعنى ذلك توزيع مباشرة هذه الاختصاصات بين الإدارة المركزية وبين الهيئات اللامركزية المستقلة محلية كانت أو مرفقية؛\nإذ تستطيع هذه الهيئات الأخيرة مباشرة اختصاصاتها استقلالاً عن السلطة المركزية، فيكون لهذه الأخيرة حق المبادأة والبت في الأمور الإدارية، وإن كان ذلك يتم بطبيعة الحال تحت رقابة السلطة المركزية في العاصمة وإشرافها.\nأما إذا اتبعت الدولة الأسلوب المركزي في توزيع السلطات الإدارية، فمعنى ذلك تركيز سلطة البت في الأمور الإدارية في يد هيئة واحدة هي السلطة المركزية في العاصمة؛\nإذ تباشر هذه السلطة الوظائف الإدارية بنفسها، وهنا نكون أمام نظام المركزية المطلقة، أو التركيز الإداري، أو تباشرها بواسطة عمالها وموظفيها المنتشرين في الأقاليم الذين يعملون باسم السلطة المركزية ولحسابها ، وهنا نكون أمام ما يطلق عليه المركزية النسبية.\nومما لاشك فيه أن التركيز الشديد للسلطة يؤدي إلى ارتباك في الجهاز الإداري، ومن ثم عجزه عن إنجاز المهمات الموكولة إليه؛\nإذ يؤدي إلى إضعاف الشعور بالمسؤولية وإضاعة الوقت والجهد والمال في إجراءات إدارية معقدة.\nللمزيد من التفاصيل حول المركزية الإدارية واللامركزية الإدارية انظر :\nد. عبد الله طلبه، مبادئ القانون الإداري، ج 1، منشورات جامعة دمشق عام 2010-2011، ص 105 وما بعدها.\nوانظر أيضاً د. نجم الأحمد ود. أحمد اسماعيل الإدارة العامة، منشورات جامعة دمشق، عام\n167\n.2016-2017، ص 137 وما بعدها [cite: 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]"
    },
    {
        "title": "الصفحة رقم 4",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nوتضعف هذه المساوئ عند اللجوء إلى عدم التركيز الذي يوفر الوقت والجهد والمال ويخلق روح الحماس لدى المرؤوسين، ويضع سلطة اتخاذ القرار في يد المرؤوسين الذين يكونون أكثر قرباً واتصالاً بالواقع وظروفه.\nوالأسلوب الأمثل للأخذ بعدم التركيز الإداري هو أسلوب التفويض الإداري.\nولا شك أن الحاجة للأخذ بأسلوب التفويض الإداري قد تزايدت في الوقت الحاضر بتزايد وظائف الدولة، واتساع حجم سلطتها التنفيذية؛\nلذلك حرصت التشريعات في كثير من دول العالم المعاصر على تقنين مبدأ التفويض، وتنظيم أحكامه كوسيلة من وسائل تحقيق سياسة عدم التركيز في المجال الإداري.\nأولاً - أهمية البحث وأهدافه:\nبعد موضوع التفويض الإداري من الموضوعات المهمة في نطاق القانون الإداري، فهو أحد الأساليب التي يمكن الاعتماد عليها في تحقيق ديمقراطية الإدارة عن طريق المشاركة في صنع القرارات الإدارية، ولا سيما أن ازدياد المهمات الإدارية وتعقدها وتشعبها تجعل من المستحيل على شخص واحد مهما كان فذاً وذكياً القيام بها وحده، فلا مناص له من الاستعانة بالآخرين، والتفويض الإداري يحقق جزءاً من تلك الغاية.\nنشير هنا إلى ضرورة التمييز بين تفويض السلطة الإدارية تفويض الاختصاص، وتفويض التوقيع، وقد رتب الفقه آثاراً مهمة على التفرقة بين هذين النوعين سنذكرها لاحقاً.\nوكذلك يختلف التفويض في السلطة الإدارية عن التفويض التشريعي الذي بواسطته تستطيع السلطة التنفيذية أن تسن التشريعات بناءً على موافقة المجلس النيابي.\nانظر في تفصيلات ذلك: د. عبد الغني بسيوني عبد الله أصول علم الإدارة العامة، الدار الجامعية، الإسكندرية عام 1992 ، ص 215 وما بعدها.\nوأيضاً د. إبراهيم عبد العزيز شيحا، أصول الإدارة العامة، منشأة المعارف، الإسكندرية عام 1993 ، ص 264 وما بعدها.\n168 [cite: 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]"
    },
    {
        "title": "الصفحة رقم 5",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nوتهدف هذه الدراسة إلى التعريف بنظرية التفويض الإداري، ودراسة أبعادها، ومحاورها المختلفة، لإلقاء الضوء على ركائزها الأساسية، وتوضيح الصعوبات والمعوقات التي تواجهها، وصولاً للتطبيق الأمثل لهذه النظرية في المجال الإداري.\nثانياً - إشكالية البحث :\nتظهر إشكالية البحث في بيان حقيقة أسلوب التفويض الإداري، ومنع تداخله مع بعض المفاهيم القانونية القريبة منه، وتحديد أهم الصعوبات التطبيقية والعملية التي تواجه عملية التفويض الإداري، وكيفية علاجها على أرض الواقع، وصولاً لتحقيق الهدف الأساسي للتفويض الإداري المتمثل بإعطاء الفرصة للموظفين بالتدريب على القيادة الإدارية، وصناعة القرار الرشيد، وسرعة البت في المسائل الإدارية، والقضاء على الروتين الحكومي والتعقيدات الإدارية.\nثالثاً - منهج البحث :\nستسلك الدراسة المنهجين التأصيلي والتحليلي، من خلال دراسة كل ما يتعلق بنظرية التفويض الإداري، والأحكام العامة الناظمة لها، وبيان الشروط العامة للتفويض الإداري وتحليل محاور البحث المختلفة، وصولاً لمعرفة أهم المعوقات العملية التي تواجه عملية التفويض الإداري، ومحاولة طرح حلول لمواجهة هذه الصعوبات تكون قابلة للتطبيق الواقعي للوصول إلى تمكين العاملين في المنظمة الإدارية من أداء مهماتهم بالصورة الأفضل والجودة الأعلى.\nرابعاً - خطة البحث:\nسأقوم في هذه الدراسة باتباع التقسيم الثنائي (اللاتيني) من خلال تقسيم الدراسة إلى مبحثينعلى النحو الآتي:\nالمبحث الأول - ماهية التفويض الإداري.\nالمبحث الثاني - مزايا التفويض الإداري وصعوباته التطبيقية وكيفية علاجها.\n169 [cite: 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]"
    },
    {
        "title": "الصفحة رقم 6",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nالمبحث الأول: ماهية التفويض الإداري :\nد. عمار التركاوي\nللوقوف على ماهية التفويض الإداري، ومعرفة حقيقته، لابد من تحديد مفهوم التفويض الإداري، وأنواعه، وبيان مبادئه وشروطه القانونية، وآثاره، والتفرقة بينه وبين ما يتشابه معه من نظم قانونية.\nوبناء على ما سبق، سنقسم دراسة هذه المبحث إلى ثلاثة مطالب على النحو الآتي:\nالمطلب الأول - مفهوم التفويض الإداري وأنواعه.\nالمطلب الثاني - مبادئ التفويض الإداري وشروطه القانونية.\nالمطلب الثالث - آثار التفويض الإداري وتمييزه من بعض النظم القانونية المشابهة له\nيقصد بالتفويض الإداري هنا تفويض السلطة أو التفويض في الاختصاصات الإدارية، انظر في تفصيلات هذا الموضوع لدى كل من د. حسين عثمان محمد عثمان أصول القانون الإداري، منشورات الحلبي الحقوقية، بيروت، عام 2008، ص 312 وما بعدها. ود.\nعبد الله طلبه الإدارة العامة، 8، منشورات جامعة دمشق عام 2007-2008 ، ص 121 وما بعدها. ود.\nعيد قريطم التفويض في الاختصاصات الإدارية لدراسة مقارنة، ط1، منشورات الحلبي الحقوقية، بيروت عام 2011، ص 22 وما بعدها. ود.\nعبد الكريم الأحمد المصطفى، المفهوم القانوني لمبدأ الإدارة بالمشاركة في نطاق علم الإدارة العامة، أطروحة دكتوراه في القانون العام، جامعة دمشق عام 2012، ص 147 وما بعدها.\nوأخيراً د. طارق المجذوب، الإدار العامة العملية الإدارية والوظيفة العامة والإصلاح الإداري ، ط1، منشورات الحلبي الحقوقية، بيروت عام 2005، ص 419 وما بعدها.\n170 [cite: 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78]"
    },
    {
        "title": "الصفحة رقم 7",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nالمطلب الأول: مفهوم التفويض الإداري وأنواعه\nسنقسم دراسة هذا المطلب إلى فرعين على النحو الآتي:\nالفرع الأول: مفهوم التفويض الإداري :\nيعرف فقهاء القانون الإداري وفقهاء الإدارة العامة التفويض الإداري: بأنه القرار الذي يتخذه الرئيس الإداري صاحب الاختصاص الأصيل، فيعهد بموجبه لأحد مرؤوسيه المفوض إليه بجزء من اختصاصاته؛\nإذ يقوم المرؤوس بمعالجتها، واتخاذ ما يلزم بشأنها من تدابير دون العودة إلى الرئيس الإداري، على أن تبقى مسؤولية إنجاز هذه الاختصاصات موزعة بين الرئيس والمرؤوس؛\nأي أن التفويض الإداري لا يعني تخلي الرئيس الإداري عن سلطاته ومسؤولياته، بل هو مجرد آلية أفضل لتقديم الخدمات، وإنجاز الأعمال بسرعة وكفاءة.\n2\nوقد أصبح التفويض الإداري اليوم هو الوسيلة الطبيعية، ولا سيما في المنظمات الإدارية الكبيرة، وهو يستخدم للتخلص من تركيز السلطة الذي يعرقل سير العمل الإداري، وللاستفادة من مزاياه المتعددة في التخفيف عن الرئيس الإداري، وتوفير الوقت والمجهود، ورفع مستوى المشاركة من جانب العاملين في المنظمة وغيرها من المزايا التي سنذكرها لاحقاً.\nانظر في تفصيلات ذلك لدى كل من د. عبد الله طلبه ود. محمد الحسين ود. مهند نوح، المدخل إلى القانون الإداري، منشورات جامعة دمشق، نظام التعليم المفتوح، عام 2012-2013، ص 113. ود. سعيد نحيلي ود. عمار التركاوي، القانون الإداري المبادئ العامة، منشورات جامعة دمشق عام 2018-2019، ص 185 وما بعدها ود. رمضان بطيخ ود. نوفان العجارمة، مبادئ القانون الإداري في المملكة الأردنية الهاشمية، ط (1) دار إثراء للنشر والتوزيع، عمان، الأردن، عام 2011، ص 249 وما بعدها و د. محمد رفعت عبد الوهاب، الإدارة العامة، دار الجامعة الجديدة، الإسكندرية عام 2007-2008 ، ص 243 وما بعدها.\nد.\nسعيد نحيلي القانون الإداري المبادئ العامة، ج (1)، منشورات جامعة البعث، عام 2012-2013\n171\n10 ص [cite: 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94]"
    },
    {
        "title": "الصفحة رقم 8",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nوبالنسبة إلى أهمية التفويض فقد ثار في البداية فهم خاطئ عن تفويض سلطة الرئيس إلى مرؤوسيه، إذ اعتقد بعضهم أن ذلك يعنى إقراراً بالضعف، وعدم استطاعة تحمل المسؤوليات؛\nأي تسليم من جانب الرئيس بعدم صلاحيته لمنصب الرئاسة الإدارية.\nوهكذا كان الرؤوساء في خوف من ضياع سلطاتهم القوية، وفي شك من استخدام التفويض، ومع مرور الزمن وتطور العلاقات الاجتماعية والنظم الإدارية أضحى التفويض هو الأسلوب الأمثل للتخفيف من التركيز الإداري، بما يحققه من مزايا تعود بالنفع على العمل الإداري.\nوعادة ما يتم تفويض السلطة على أساس تحليل كامل للوظيفة الإدارية، من تنظيم سير العمل، ووضع السلطات في الجهاز الإداري كله، وإعطاء الحرية اللازمة لاتخاذ القرارات الضرورية دون عراقيل .\nكما أنه توجد عدة عوامل تؤثر في درجة تفويض السلطة من أهمها : خطورة القرار، ومدى ما يترتب عليه من أعباء مالية، والرغبة في اتخاذ سياسة موحدة للمنظمة الإدارية في مجموعها، ومدى توافر القادة الإداريين، وأيضاً طرق الرقابة المستخدمة.\nويعتقد فقهاء الإدارة العامة أنه مهما زاد النجاح في عملية التفويض على المستويات المختلفة، فهناك سلطات معينة ينبغي للرئيس الاحتفاظ بها، وعدم تفويضها، وتشمل هذه السلطات : 2\nاد. عبد الغنى بسيونى عبد الله، مرجع سبق ذكره، ص 216.\nد. عبد الله طلبه الإدارة العامة، مرجع سبق ذكره، ص 124 وما بعدها. ود. محمود عساف أصول الإدارة مكتبة لطفي القاهرة عام 1988 ، ص 376 وما بعدها. ود. محمد رفعت عبد الوهاب، مرجع سبق ذكره ص 254 وما بعدها.\n172 [cite: 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107]"
    },
    {
        "title": "الصفحة رقم 9",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\n1. المسائل المالية والتصرف في الموازنة.\n2. القرارات الكبرى المتعلقة بالتشريع داخل المنظمة، وخارجها.\n3. التعديل والتغيير في السياسة العامة للمنظمة الإدارية، والتغييرات الكبرى في طرق العمل، وأساليبه، وإجراءاته.\n4. التعيين في الوظائف القيادية الرئيسة في المنظمة.\nالفرع الثاني: أنواع التفويض الإداري\nيميز الفقه بين نوعين من التفويض الإداري هما : تفويض السلطة أو تفويض الاختصاص) وتفويض التوقيع. ويرتب الفقه على هذه التفرقة النتائج الآتية:\n1. إن تفويض السلطة يحرم المفوّض من ممارسة الاختصاصات بجوار المفوض إليه طيلة مدة التفويض؛\nلأن هذا النوع من التفويض يؤدي إلى نقل الاختصاصات المفوض فيها.\nأما التفويض في التوقيع فلا يحرم المفوّض من حق التوقيع بجوار المفوض إليه، طيلة فترة التفويض.\n2. إن القرار الذي يصدر بناءً على تفويض الاختصاص ينسب إلى المفوض إليه، ويستمد قوته من مركزه في السلم الإداري، بينما ينسب القرار الصادر في نطاق تفويض التوقيع إلى المفوّض نفسه، ويأخذ قوة القرارات الصادرة عنه.\n3. يوجه التفويض في الاختصاص إلى الشخص بصفته لا بشخصه، كتفويض الوزير أو المحافظ مثلاً، ويترتب على ذلك أن التفويض لا ينتهي إذا تغير شخص المفوض إليه.\nاد. عبد الله طلبه القانون الإداري الرقابة القضائية على أعمال الإدارة منشورات جامعة دمشق، عام 2017-2016، ص 268 . ود. ابرهيم عبد العزيز شيحا، مرجع سبق ذكره، ص 281 وما بعدها. ود. مصطفى أبو زيد فهمي، القضاء الإداري ومجلس الدولة، القاهرة عام 1979، ص 470. ود. ماجد راغب الحلو، القانون الإداري، دار الجامعة الجديدة، الإسكندرية، عام 2006، ص 85 وما بعدها.\n173 [cite: 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125]"
    },
    {
        "title": "الصفحة رقم 10",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nأما التفويض في التوقيع فهو شخصي، ومن ثم فهو ينقضي إذا ما تغير شخص المفوض أو المفوض إليه.\n1\nالمطلب الثاني: مبادئ التفويض الإداري وشروطه القانونية:\nإن أي نظام بشري - ومهما كانت فوائده يجب أن يكون مضبوطاً بمجموعة من القواعد القانونية والإدارية، كي لا يساء استخدامه، وينقلب إلى عائق في طريق إنجاز المهمات الإدارية.\nوهناك إجماع فقهي على أن التفويض الإداري تحكمه مجموعة من المبادئ والشروط القانونية، وسنقوم بتوضيح ذلك في فرعين على النحو الآتي:\nالفرع الأول: مبادئ التفويض الإداري:\nأولاً - التفويض لا يكون إلا جزئياً : أي أن التفويض يجب ألا يشمل كل اختصاصات الأصيل؛\nلأن تفويض كامل الاختصاصات يعني نقل الاختصاص إلى المرؤوس، وهذا أمر غير جائز ؛\nلأن الأصل أن يمارس الرئيس الإداري الذي حدده المشرع الاختصاصات\nيرى بعض الفقه أن تفويض التوقيع وتفويض الاختصاص يشتركان في كثير من النقاط لجهة أن كلا منهما بعد أداة لتحقيق عدم التركيز الإداري، ونحن بدورنا نشاطر هذا الرأي ونضيف أن الغرض الأساسي لتفويض التوقيع لا يعدو أن يكون تخفيف العبء المادي عن الرئيس الإداري، وسرعة إنجاز المعاملات، أما تفويض الاختصاص فيهدف لخلق جسور الثقة بين الرئيس والمرؤوس، وخلق صف ثان جاهز للعمل القيادي.\nانظر د. سعيد نحيلي ود. عمار التركاوي، مرجع سبق ذكره، ص 194 وما بعدها.\nانظر في تفصيلات ذلك د. محمد الحسين ود. مهند نوح القانون الإداري عمال الإدارة العامة وتصرفاتها القانونية، منشورات جامعة دمشق عام 2011-2012، ص 154 و د. سعيد نحيلي ود. عمار التركاوي مرجع سبق ذكره، ص 188 وما بعدها. ود. عبد الغني بسيوني عبد الله، مرجع سبق ذكره، ص 217 وما بعدها.\nوأيضاً شادي إسماعيل اللوائح التنفيذية والرقابة القضائية عليها - دراسة مقارنة ، رسالة ماجستير في القانون العام، جامعة دمشق عام 2013، ص 164.\n174 [cite: 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142]"
    },
    {
        "title": "الصفحة رقم 11",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nالمقررة له، والاستثناء أن يلجأ إلى تفويض جزء من اختصاصاته لأحد مرؤوسيه إذا أذن المشرع بذلك.\nثانياً - عدم جواز تفويض الاختصاصات المفوضة: يعني هذا المبدأ أنه لا يجوز للمرؤوس المفوض إليه أن يقوم بإعادة تفويض السلطات التي انتقلت إليه لمن هم أدنى منه في السلم الوظيفي؛\nلأن عملية التفويض الإداري لا تتم إلا مرة واحدة، أما أن يقوم المرؤوس المفوض إليه بتفويضها إلى من يليه في السلم الإداري فهذا الأمر مرفوض؛\nلأنه سيشيع المسؤولية، ويسبب عرقلة للعمل الإداري، وهذا الأمر يفرغ التفويض من فوائده.\nثالثا - ينصب التفويض على السلطة دون المسؤولية : يشمل اصطلاح الاختصاص في القانون الإداري على شقين : السلطة والمسؤولية.\nوكقاعدة عامة فإن التفويض ينصب على السلطة فقط دون المسؤولية؛\nأي أن الرئيس الإداري يقوم بتفويض جانب من سلطاته بصفة مؤقتة، مع بقاء مسؤوليته كاملة.\nإذ إن مسؤولية الرئيس الإداري أمام الرئاسات العليا التابع لها لا يمكن أن تنتقل مع تفويض بعض اختصاصاته إلى من هم أقل منه في السلم الإداري، فالتفويض لا يعني تخلي الرئيس الإداري عن سلطاته ومسؤولياته، وانما هو وسيلة لتوزيع السلطة، والقضاء على تركيزها من أجل مصلحة العمل في المنظمة الإدارية.\nوطبقاً لمبدأ تناسب السلطة مع المسؤولية فإن المفوض إليه يكون مسؤولاً أمام الرئيس المباشر بقدر السلطة التي انتقلت إليه عن طريق التفويض، ومن ثم فإن مسؤولية الرئيس الإداري الأصيل تبقى محصورة في الحالات التي يثبت فيها إهمال أو تقصير في متابعة أعمال المرؤوسين، أو إذا ثبت أن هناك لا مبالاة، أو سوء نية في اختيار المفوض إليه.\nرابعاً - التفويض لا يكون إلا من أعلى إلى أسفل: وهذا شرط بديهي، وينطلق من طبيعة التفويض نفسه ؛\nلأن التفويض كما ذكرنا - وسيلة للتخلص من التركيز الشديد في السلطات في الهرم الإداري، فيلجأ الرئيس إليه وينتقل إلى من هم أدنى منه جانباً من تلك\n175 [cite: 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]"
    },
    {
        "title": "الصفحة رقم 12",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nالاختصاصات، ولا يمكن حدوث العكس، فمن غير المتصوّر أن يتم التفويض من المرؤوس إلى رئيسه.\nخامساً - وضوح حدود التفويض يجب على الرئيس الإداري أن يعين حدود التفويض بدقة ووضوح حتى لا يقع أي نزاع أو سوء فهم أثناء ممارسة السلطات المفوضة بين المرؤوسين الذين انتقلت إليهم تلك السلطات.\nسادساً - حق الرئيس الإداري في تعديل السلطات المفوضة أو استردادها: بعد نقل السلطات المفوضة إلى المرؤوس لا يجوز للرئيس الإداري أن يمارس تلك السلطات؛\nلأنها أصبحت من حق المرؤوس من ناحية، ولأنه لو كان للرئيس الإداري حق ممارسة الاختصاصات المفوضة لحدث تعارض في القرارات الصادرة من المفوّض والمفوض إليه في موضوع الاختصاص نفسه.\nولكن هذا لا يمنع من قيام الرئيس بمراقبة مرؤوسه، وتوجيهه وإرشاده إلى الكيفية التي يمارس بها تلك السلطات.\nوإذا شاء الرئيس أن يعدل في نطاق وحدود السلطات المفوضة بناءً على تلك المراقبة، فله الحق في ذلك، وفي النهاية يستطيع الرئيس الإداري سحب السلطات المفوضة، والغاء التفويض إذا ما رأى ذلك.\nويحدث هذا غالباً عند إعادة تنظيم الجهاز الإداري؛ إذ تعود السلطات المفوضة إلى الأصيل، ثم تجري عملية تفويض جديدة عند إعادة توزيع الاختصاصات.\nوقد يسترد الرئيس السلطات المفوضة إذا ما وجد أن المرؤوس قد أساء استخدام السلطة المفوضة إليه، وأنه استخدم هذه السلطة بطريقة لا تعود بالنفع على المنظمة الإدارية.\n1\nد. عبد الغني بسيوني عبد الله، مرجع سبق ذكره، ص 219-220\n176 [cite: 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169]"
    },
    {
        "title": "الصفحة رقم 13",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nالفرع الثاني: الشروط القانونية لصحة التفويض الإداري:\nيرتبط التفويض الإداري بمجموعة من الشروط القانونية نوضحها في النقاط الآتية: 1\nأولاً- وجود إذن تشريعي:\nيجب لصحة التفويض الإداري وجود نص تشريعي يسمح للرئيس الإداري التفويض في بعض اختصاصاته، ويأخذ هذا الإذن عادة إحدى صورتين\nفإما أن ينص التشريع الخاص الذي ينظم اختصاصات جهة إدارية ما على إجازة التفويض في موضوعات يحددها المشرع نفسه، ويجيز التفويض فيها.\nوإما أن يصدر تشريع عام ينظم عملية التفويض الإداري في الدولة.\nومن أمثلة الصورة الأولى نذكر ما ورد في نص المادة / 55 / ف / 2/ من قانون الإدارة المحلية الصادر بالمرسوم التشريعي رقم 107/3/ لعام 2011، فقد جاء فيها: \"للمحافظ بصفته رئيساً للمكتب التنفيذي أن يفوّض نائب رئيس المكتب أو الأمين العام أو مديري الأجهزة المحلية والمركزية ببعض اختصاصاته وفق القوانين والأنظمة\".\nوكذلك ما ورد في المادة / 6/ من قانون تنظيم الجامعات رقم /6 / لعام 2006 من أنه \"الرئيس الجامعة أن يعهد إلى أي من نوابه بدراسة ما يراه من الموضوعات أو البت فيها، كما له أن يفوض بقرار منه إلى أي منهم، في حدود اختصاصاته، أو البت بصورة دائمة في موضوعات معينة، ويُعمّم التفويض في هذه الحالة، وتُبلغ الوزراة صورة عنه\".\nكذلك نجد المادة 13/1/ من القانون نفسه تنص على أنه: \" لعميد الكلية أن يعهد إلى أي من نائبيه بدراسة ما يراه من الموضوعات، أو البت فيها ، كما له أن يفوض إلى أي\nللنظر في تفصيلات هذه الشروط راجع د. سعيد نحيلي ود. عمار التركاوي، مرجع سبق ذكره، ص 191 وما بعدها. ود. ابراهيم عبد العزيز شيحا، مرجع سبق ذكره، ص 274 وما بعدها. ود. عبد الغني بسيوني عبدالله مرجع سبق ذكره، ص 244 وما بعدها.\n177 [cite: 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182]"
    },
    {
        "title": "الصفحة رقم 14",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nمنهما - في حدود اختصاصاته - أمر البت بصورة دائمة في موضوعات معينة، ويجب تعميم التفويض في هذه الحالة على أقسام الكلية، وتبليغ صورة عنه إلى رئاسة الجامعة\".\nأما مثال الصورة الثانية التفويض بموجب نص عام فهو ما ورد في أحكام المرسوم التشريعي رقم / 69 / لعام 2005 في المواد c1(5-4-3-2-1) إذ يعد هذا المرسوم إذناً تشريعياً لكل رئيس إداري يقوم على تنفيذ القوانين والأنظمة التي تخص الجهة العامة التي يقودها بأن يفوض بعض اختصاصاته لمرؤوسيه، ومن ثم لم يعد هناك أي ذريعة لدى الرؤساء الإداريين الذين يحجمون عن إجراء التفويض الإداري بداعي أن القوانين والأنظمة لا تسمح بذلك.\nونشير أيضاً إلى أن دستور الجمهورية العربية السورية الحالي لعام 2012 سمح لرئيس الجمهورية في المادة / 91/ منه على أن يسمي نائباً له أو أكثر، وأن يفوضهم ببعض صلاحياته.\nونتساءل عن مصير القرارات التي يتخذها المفوض إليه بناءً على تفويض إداري من رئيسه دون أن يأذن المشرع بهذا التفويض؟\nيجمع الفقه على بطلان هذه القرارات لعدم استنادها إلى أساس قانوني سليم؛\nلأنها بنيت على تفويض باطل، عملاً بالقاعدة الفقهية ما بني على باطل فهو باطل\".\nثانياً - أن يكون النص القانوني الآذن بالتفويض من ذات مرتبة النص الذي خول الاختصاص للأصيل نفسه\nوعلى ذلك فإذا كان الاختصاص الأصيل مبناه نص في الدستور، فإن التفويض لا يكون جائزاً إلا إذا سمح به نص دستوري، أي نص قانوني له القوة القانونية نفسها للنص السابق\nالمرسوم التشريعي رقم 69 لعام 2005 الناظم لتفويض الاختصاص في سورية ألغى العمل بأحكام القانون رقم /52/ لعام 2001 الذي كان ينظم نفس الموضوع.\nد. سعيد نحيلي ود. عمار التركاوي، مرجع سبق ذكره، ص 192.\n178 [cite: 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196]"
    },
    {
        "title": "الصفحة رقم 15",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nالذي خوّل الاختصاص الأصيل ؛ لذا لا يجوز لرئيس الجمهورية عند تفويض جزء من اختصاصاته الدستورية إلى أحد نوابه الاستناد إلى المرسوم التشريعي رقم 69/ لعام 2005 لأن هذا النص أقل مرتبة من الدستور.\nثالثاً - صدور قرار بالتفويض من صاحب الاختصاص الأصيل:\nلا يكفي لقيام التفويض مجرد الإذن التشريعي، وتحديد شخص المفوّض، بل لابد من صدور قرار من الرئيس المفوض صاحب الاختصاص الأصيل، يتم من خلاله تفعيل النص القانوني، ونقله إلى حيز التنفيذ.\nرابعاً - يجب أن يكون التفويض كتابياً :\nوقد أقر الاجتهاد القضائي بعدم الاعتداد بالتفويض إذا كان شفهياً، فالعبرة هي لكتابة التفويض، وتبليغه للمخاطبين بأحكامه .\nالمطلب الثالث: آثار التفويض الإداري وتمييزه من بعض النظم القانونية المشابهة له\nللتفويض الإداري آثار متعددة، كما أنه يتداخل مع بعض المفاهيم القانونية الأخرى، وسنقوم بدراسة هذا المطلب على النحو الآتي:\nلمزيد من التفصيلات حول هذا الموضوع انظر : د. محمد باهي أبو يونس، الضوابط الدستورية للوظيفة اللائحية التنفيذية - دراسة مقارنة، دار الجامعة الجديدة، الإسكندرية عام 2008، ص 54 وما بعدها.\nفقد قضت محكمة القضاء الإداري المصرية في حكم لها صادر بتاريخ 21 أبريل عام 1949 بأنه:\" لا يلتفت إلى القول بصدور تفويض شفوي من مجلس الوزراء إلى وزير المالية في شأن وقف العمل بقواعد مجلس الوزراء، لأن مثل هذا التفويض لا يكون إلا بقرار يصدره مجلس الوزراء بالطرق المعتادة ثم يبلغه إلى وزارة المالية. \" مجموعة المبادئ القانونية التي قررتها محكمة القضاء الإداري المصرية، السنة الثامنة، حكم / 43، ص 250.\n179 [cite: 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208]"
    },
    {
        "title": "الصفحة رقم 16",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nالفرع الأول: آثار التفويض الإداري\nللتفويض آثار سواء بالنسبة إلى الإداري المفوض أم بالنسبة إلى المرؤوس الذي فوض إليه الاختصاص.\nفبالنسبة إلى الرئيس الإداري الذي فوض بعض سلطاته، فإنه يظل مسؤولاً عن أداء المفوض إليه للسلطات والاختصاصات الصادر بها التفويض.\nفمثلاً لو فوض وزير الصحة أحد معاونيه الإشراف على شؤون الصحة المدرسية وعلاج الطلاب، فإن هذا الوزير لا يستطيع أن يدفع مسؤوليته أمام رئيس الجمهورية، أو أمام مجلس الشعب عن أمور خاصة بالصحة المدرسية وعلاج الطلاب، أنه قد فوض هذه الأمور إلى أحد معاونيه، ومع ذلك فمسؤوليته - مع التفويض - تظل قائمة؛\nلأنه كما ذكرنا سابقاً المسؤولية لا تُفوّض؛ لذلك قلنا إن التفويض يستلزم ضرورة المتابعة والتقييم من جانب الرئيس المفوض، كما أن لهذا الأخير حق سحب التفويض وتعديله، كما أن من حقه سحب القرارات الصادرة من المفوض إليه، أو إلغائها، أو تعديلها.\nأما بالنسبة إلى المرؤوس الذي فوض إليه الاختصاص فإنه يلاحظ أن التفويض يخلق التزاماً على عاتقه تجاه الرئيس الذي فوضه الاختصاص مقتضاه إنجاز العمل من خلال السلطة التي أعطيت له بموجب قرار التفويض.\nوإذا رفض المفوض إليه العمل بمقتضى قرار التفويض، فيعد هذا التصرف مرتباً المسؤوليته المسلكية، وهذا يبرر توقيع الجزاء التأديبي عليه.\n2\nاد. عبد الفتاح حسن، مبادئ الإدارة العامة، القاهرة عام 1972 ، ص 102.\nد. إبراهيم شيحا، مرجع سبق ذكره، ص 271 وما بعدها. ود. محمد رفعت عبد الوهاب، مرجع سبق ذكره، ص 260 وما بعدها .\n180 [cite: 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222]"
    },
    {
        "title": "الصفحة رقم 17",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nالفرع الثاني: تمييز التفويض الإداري عن بعض النظم القانونية المشابهة له\nأولاً - التفويض الإداري والحلول : 1\nذكرنا أن التفويض الإداري هو أن يعهد الرئيس الإداري بممارسة جانب من اختصاصاته التي يستمدها من القانون إلى فرد آخر وفقاً للشروط القانونية المقررة لذلك، أما الحلول فهو انتقال جميع اختصاصات الأصيل، في حال قيام مانع يحول دون ممارسته لها إلى موظف آخر حدده المشرع مسبقاً بالنص القانوني الناظم لعمل المنظمة ذات الصلة، بممارسة اختصاصات الأصيل الغائب إلى حين عودة هذا الأخير، فالحلول هو وسيلة مناسبة لسد الفراغ الذي ينجم عن غياب الأصيل لأي سبب كان.\nومن الأمثلة التي يمكن أن نذكرها عن الحلول ما ورد في المادة /93/ من دستور الجمهورية العربية السورية لعام 2012، فقد نصت على أنه:\n1. في حالة شغور منصب رئيس الجمهورية أو عجزه الدائم عن أداء مهامه، يتولى مهامه مؤقتاً النائب الأول لرئيس الجمهورية لمدة لا تزيد عن تسعين يوماً من تاريخ شغور منصب رئيس الجمهورية، على أن يتم خلالها إجراء انتخابات رئاسية جديدة.\n2. في حالة شغور منصب رئيس الجمهورية ولم يكن له نائب، يتولى مهامه مؤقتاً رئيس مجلس الوزراء لمدة لا تزيد عن تسعين يوماً من تاريخ شغور منصب رئيس الجمهورية، على أن يتم خلالها إجراء انتخابات رئاسية جديدة.\"\nانظر في تفصيلات ذلك لدى كل من د. ماجد راغب الحلو القانون الإداري، مرجع سبق ذكره، ص 85 وما بعدها.\nود. سعيد نحيلي ود. عمار التركاوي، مرجع سبق ذكره، ص 198 وما بعدها. ود. محمد رفعت عبد الوهاب مرجع سبق ذكره، ص 261 وما بعدها و د. ماجد راغب الحلو، علم الإدارة العامة ومبادئ الشريعة الإسلامية، دار الجامعة الجديدة، الإسكندرية، عام 2007، ص 309 وما بعدها.\n181 [cite: 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233]"
    },
    {
        "title": "الصفحة رقم 18",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nوهناك مثال آخر في قانون الإدارة المحلية الصادر بالمرسوم التشريعي رقم /107/ لعام 2011، إذ تنص المادة 1/6/55/ منه على أنه في حال غياب المحافظ ينوب عنه قائد شرطة المحافظة بوصفه ممثلاً عن السلطة التنفيذية\". وهذا مثال واضح عن الحلول مع أن المشرع استخدم كلمة (ينوب) التي لا تغير من النتيجة شيئاً؛ لأن شروط الحلول قد توافرت كلياً.\nفإذا أصدر نائب رئيس المكتب التنفيذي (الحال) قراراً إدارياً نجم عنه مساس بالمركز القانوني لأحد المواطنين بصورة غير مشروعة، فمن حق هذا المواطن رفع دعوى إلغاء القرار، وفي هذه الحالة يكون المدعى عليه المحافظة يمثلها المحافظ، وكأن القرار صدر عنه ابتداء.\nوبناءً على ما تقدم، يمكن إجمال أهم الفروق القائمة بين التفويض الإداري والحلول بالنقاط الآتية : 2\n1. التفويض الإداري عمل إرادي، إذ يتوقف على إرادة المفوّض، فهو الذي يحدد وقت التفويض، وشخص المفوض إليه، بل يستطيع سحب التفويض أو إلغاءه في أي وقت يشاء.\nأما الحلول فيتم بقوة القانون عند تحقق سببه، دون أي تدخل من جانب الأصيل.\nفالحلول إذاً لا يترك لهذا الأخير أي سلطة لا في اختيار وقت الحلول، ولا في اختيار شخص الحال.\n2. التفويض الإداري ينصب على بعض اختصاصات المفوّض، أما الحلول فهو كلى، أي يتناول جميع اختصاصات الأصيل الغائب.\nاد. سعيد نحيلي ود. عمار التركاوي، مرجع سبق ذكره، ص 200 وما بعدها.\nد. رمضان بطيخ ود. نوفان العجارمة، مرجع سبق ذكره، ص 255 وما بعدها. ود. محمد رفعت عبد الوهاب، مرجع سبق ذكره، ص 262 وما بعدها.\n182 [cite: 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248]"
    },
    {
        "title": "الصفحة رقم 19",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\n3 في التفويض الإداري لا يمارس المفوض إليه من الاختصاصات إلا ما فوض إليه، وهو يخضع في ممارستها لسلطة الأصيل الرئاسية، ذلك الأصيل الذي يظل مع ذلك مسؤولاً عن نتائج تلك الممارسة.\nأما في الحلول - إذ يكون للشخص الحال ذات اختصاصات وسلطات الأصيل بقوة القانون - فإنه يترتب على ذلك أن القرارات الصادرة بمناسبة الحلول تكون لها مرتبة وقوة القرارات الصادرة من الأصيل، ومن جهة أخرى فإن الحال لا يخضع في مباشرته لاختصاصات الأصيل للسلطة الرئاسية لهذا الأخير، بل تقع عليه وحده المسؤولية عن تصرفاته.\n1\n4. ينتهي التفويض الإداري نهاية طبيعية بانتهاء مدته إذا كان محدد المدة، أو بصدور قرار إداري من الأصيل يلغي بموجبه مفاعيل قرار التفويض.\nأما الحلول فينتهي بعودة الأصيل الغائب آلياً، وبقوة القانون أي دون الحاجة إلى صدور قرار من أحد، أو بصدور قرار من السلطة المختصة بتكليف أو تسمية شخص جديد على رأس المنظمة.\nثانياً - التفويض الإداري والإنابة:\nيُقصد بالإنابة أن تصدر جهة إدارية قراراً بتعيين أحد الأشخاص لممارسة اختصاصات شخص آخر تغيب لسبب من الأسباب عن مزاولة الأعمال المنوطة به، وفي الإنابة نكون أمام ثلاثة أطراف:\nالطرف الأول: وهو المنيب؛ أي الجهة المصدرة لقرار الإنابة، وهي جهة رئاسية لكل من الأصيل والنائب.\nمع ملاحظة أن هذه الجهة يجب أن تستند عند إصدارها لهذا القرار إلى نص تشريعي يجيز الإنابة.\nالطرف الثاني: هو الأصيل الذي تغيب عن ممارسة اختصاصاته لسبب من الأسباب.\nاد. سامي جمال الدين الوسيط في دعوى إلغاء القرارات الإدارية، ط1، منشأة المعارف، الإسكندرية، عام 2004، 15 ص\n183 [cite: 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263]"
    },
    {
        "title": "الصفحة رقم 20",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nالطرف الثالث: هو النائب الذي صدر له قرار بمباشرة اختصاصات الأصيل.\nومن الأمثلة عن الإنابة ما نصت عليه المادة / 31 من قانون تنظيم الجامعات 6/29 لعام 2006 على أنه وفي حال غياب رئيس الجامعة يُكلف بمهامه أحد نوابه بقرار من الوزير .\nفنحن هنا أمام إنابة؛ لأن المشرع لم يحدد ابتداءً من يحل محل رئيس الجامعة لممارسة اختصاصاته في أثناء غيابه، بل أسند هذه المهمة إلى وزير التعليم العالي الذي يقوم بإصدار قرار إداري يسمّي بموجبه أحد نواب رئيس الجامعة ليتولى اختصاصات هذا الأخير طيلة فترة غيابه.\nوبنظرة متأنية في ماهية الإنابة، والتفويض الإداري، نجد أنهما يتشابهان في ضرورة استناد القرار الصادر بكل منهما إلى نص قانوني، وأن النائب والمفوض إليه يمارسان اختصاصات الأصيل بصفة مؤقتة، إلا أنهما يختلفان في عدة أمور منها:\n1. يمارس النائب في أغلب الأحوال جميع اختصاصات الأصيل، بينما لا يكون التفويض في الاختصاص إلا جزئياً.\n2. يصدر قرار الإنابة من جهة أخرى غير الأصيل، بينما قرار التفويض لا يصدر إلا من الأصيل نفسه.\n3. يخضع المفوض إليه عند ممارسته للاختصاصات المفوضة للسلطة الرئاسية للأصيل، وهذا الأصيل يكون مسؤولاً عن تصرفاته.\nأما في حالة الإنابة فإن النائب يحتل مرتبة الأصيل، ويكون بالتالي مسؤولاً عن أعماله.\n4. ترتبط قوة القرارات الصادرة في حالة التفويض بدرجة المفوض إليه في السلم الإداري.\nأما القرارات الصادرة في حالة الإنابة فإنها تأخذ المستوى الوظيفي نفسه لقرارات الأصيل.\n5. الإنابة تفترض غياب الأصيل، فلا وجود للأصيل مع النائب، أما في حالة التفويض فإن الأصيل موجود ويمارس اختصاصاته بجانب المفوض إليه.\n184 [cite: 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277]"
    },
    {
        "title": "الصفحة رقم 21",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nالمبحث الثاني: مزايا التفويض الإداري وصعوباته التطبيقية وكيفية علاجها :\nيحقق التفويض الإداري العديد من الفوائد في المجال الإداري؛\nلما يتسم به من مزايا جعلت منه أسلوباً ناجحاً في الحد من تركيز السلطة.\nوبالمقابل يواجه هذه الأسلوب الإداري عدداً من الصعوبات العملية، والمعوقات التي قد تؤدي إلى عدم إعماله، أو إلى التقليل من فاعليته، وهذا الأمر يسلتزم العمل على تذليل هذه العقبات وعلاجها، حتى يتحقق الهدف الذي يسعى التفويض الإداري للوصول إليه.\nوسنقوم بدراسة هذه المبحث من خلال تقسيمه إلى المطالب الآتية:\nالمطلب الأول - مزايا التفويض الإداري\nالمطلب الثاني - الصعوبات التطبيقية للتفويض الإداري\nالمطلب الثالث - علاج الصعوبات التي تعترض التفويض الإداري\nالمطلب الأول: مزايا التفويض الإداري\nأصبح التفويض الإداري ضرورة من ضرورات حسن تنظيم العمل الإداري في كل منظمة إدارية، وتعود هذه الأهمية المتزايدة للتفويض إلى المزايا والفوائد التي تعود على المنظمة من تطبيقه ، وهذا ما سنحاول توضيحه من خلال الفروع الآتية:\nالفرع الأول: سرعة إصدار القرارات والقضاء على البطء في الإجراءات\nتتجلى أولى مزايا التفويض الإداري في القضاء على الانتقاد الرئيسي الموجه إلى تركيز السلطة المتمثل في البطء الشديد في اتخاذ القرارات وفي الإجراءات .\nفالتفويض يسمح بصدور القرارات بسرعة لمواجهة الوقائع حين حدوثها، وحل المشكلات قبل أن تتعقد، ويرجع ذلك\nانظر في هذه المزايا لدى كل من د. رمضان بطيخ ود. نواف العجارمة، مرجع سبق ذكره، ص 250 وما بعدها، ود. طارق المجذوب مرجع سبق ذكره، ص 421. وأيضاً د. عبد الغني بسيوني عبد الله، مرجع سبق ذكره، ص 220 وما بعدها.\n185 [cite: 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291]"
    },
    {
        "title": "الصفحة رقم 22",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nإلى قيام المفوض إليه بإصدار القرار لمواجهة الظروف الطارئة، أو لدفع العمل في سياقه الطبيعي من دون عرقلة، أو تأخير ، ومن دون الرجوع إلى رئيسه المباشر.\nوتبدو أهمية هذه الميزة في التفويض الخارجي الذي يتم من الوزير إلى رؤساء الإدارات التابعة لوزارته في المحافظات، إذ يحقق ذلك سرعة كبيرة في البت في الأمور الإدارية، وفي إصدار القرار في الوقت المناسب.\nوغني عن البيان أن السرعة في اتخاذ القرار تحقق فائدة مهمة تتمثل في رفع قيمة القرار الذي يكون عادة قراراً رشيداً يعالج المشكلة بصورة حاسمة لصدوره من واقع الظروف المحيطة به، إذ تتجمع لدى موظف الإدارة الفرعية المعلومات والبيانات الصحيحة نتيجة لاتصاله المباشر مع الجمهور.\nالفرع الثاني : تفرغ الرئيس الإداري للمهام القيادية\nيستطيع الرئيس الإداري أن يخفف من الأعباء الإدارية (الروتينية التي تعرقله عن التفرغ لمهماته القيادية عن طريق التفويض الإداري.\nوبذلك تنتقل سلطة البت في المسائل اليومية العادية إلى المرؤوسين، ويتفرغ الرئيس الإداري للأمور المهمة المتصلة بمنظمته، ويركز جهوده في التوجيه والإشراف والتنسيق ورسم السياسة العامة للمنظمة، ومتابعة تحقيق أهدافها الأساسية.\nالفرع الثالث: خفض التكاليف المالية للقرارات الإدارية\nتتضمن التكلفة المالية للقرار الإداري قيمة المواد المستخدمة، واستهلاك الآلات والأجهزة وإيجار الأماكن من ناحية، وأجور العاملين عن ساعات العمل التي أنفقوها في عملية تحضير القرار، وإصداره من ناحية أخرى.\nولا شك أن تفويض سلطة اتخاذ هذا القرار للموظف الذي تلقى طلب المواطن مباشرة دون الرجوع إلى رئيس المنظمة يؤدي إلى خفض كبير في تكلفة القرار بعناصره السابقة.\n186 [cite: 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303]"
    },
    {
        "title": "الصفحة رقم 23",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\nوعلى العكس من ذلك فإن عدم التفويض وتركيز السلطة في قيادة المنظمة يؤدي إلى ارتفاع التكلفة المالية للقرار ؛\nلأن الموظف في الفرع عند تلقيه الطلب سيقوم بمعاونة موظفي الفرع الآخرين بكتابة التقارير، وإعداد الدراسات، وتجميع الوثائق، تمهيداً لرفع ذلك لرئيس المنظمة حتى يستطيع هذا الأخير البت في الطلب، وإصدار القرار .\nالفرع الرابع: خلق قيادات جديدة قادرة على تحمل المسؤولية 2\nينتج التفويض الإداري العديد من الآثار الطيبة لدى المرؤوسين، إذا يتم إشراكهم في إصدار القرارات الإدارية المسيرة لنشاط المنظمة الإدارية، وهذا الأمر يمنحهم الشجاعة والقدرة على تحمل المسؤولية، ويولد في نفوسهم روح الاهتمام، والحماس الكبير لتحقيق أهداف المنظمة، ويعزز ثقتهم بأنفسهم وبإمكانياتهم.\nولا شك أن هذه الآثار الإيجابية التي تخلقها عملية التفويض الإداري لدى العاملين في المنظمة الإدارية، تساعد على نمو القدرات القيادية لدى العديد من المرؤوسين، وتخلق بالتالي طبقة جديدة تكون مؤهلة لتحمل المسؤوليات الكبرى والاضطلاع بأعباء القيادة الإدارية مستقبلاً.\nالمطلب الثاني الصعوبات التطبيقية للتفويض الإداري :\nقد يعتقد بعضهم أنه بعد تحديد مضمون التفويض، وبيان حدوده، وتوضيح نطاقه ستجري عملية التفويض في العمل الإداري بيسر وسهولة، ومن دون أن تعترضها مشكلات أو صعوبات.\nاد. محمد رفعت عبد الوهاب، مرجع سبق ذكره، ص 264 وما بعدها.\nد. رمضان بطيخ ود. نواف العجارمة، مرجع سبق ذكره، ص 250 ود. عبد الغني بسيوني عبدالله، مرجع سبق ذكره، ص 223 ود. ابراهيم شيحا مرجع سبق ذكره، ص 266\n187 [cite: 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315]"
    },
    {
        "title": "الصفحة رقم 24",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nولكن الحقيقة عكس ذلك، فعلى أرض الواقع تقابل التفويض الإداري كثير من المشكلات العملية، وقد تعترضه العديد من الصعوبات والمعوقات، ما يتسبب في ظهور التعقيدات الإدارية التي تؤثر في سير العمل الإداري، وتؤدي إلى زعزعة الثقة في المزايا التي يحققها التفويض.\nوسنقوم بدراسة هذه الصعوبات التطبيقية (العملية) من خلال الفرعين الآتيين:\nالفرع الأول: الصعوبات التي ترجع إلى الرؤساء الإداريين\nيلاحظ أن بعض الرؤساء الإداريين يترددون في تفويض بعض اختصاصاتهم إلى مرؤوسيهم، وهناك مجموعة من الأسباب تدفع مثل هؤلاء الرؤساء إلى هذا الموقف السلبي من أهمها :\n1. روح الأنانية وحب الذات الذي يسيطر على كثير من الرؤساء الإداريين، والذي يدفعهم في أغلب الأحيان إلى الاستئثار بأعمال المنظمة الإدارية جميعاً ، مهما كان نقل هذه الأعمال.\n2. انعدام الثقة في المرؤوسين، وعدم الاطمئنان إلى قيامهم بالعمل الموكول إليهم على أحسن وجه.\nوينتج عن ذلك أن يشكك الرئيس الإداري دائماً في مقدرة مرؤوسيه على الاضطلاع بأعمالهم كما ينبغي، وقيامهم بإصدار القرارات الإدارية السليمة المحققة للمصلحة العامة.\n3 الخوف من تمرس المرؤوسين على العمل الإداري، وتدربهم على إصدار القرارات والمامهم بدقائق العمل في المنظمة الإدارية.\nوهذا الخوف ناتج من الاعتقاد من جانب بعض الرؤساء بأن اكتساب الخبرة سيدفع المرؤوسين إلى الاستهانة برؤسائهم، وتفشي روح عدم الولاء والطاعة بينهم.\nاد. رمضان بطيخ ود. نواف العجارمة، مرجع سبق ذكره، ص 261 وما بعدها و د. ابراهيم شيحا، مرجع سبق 267 ذكره، ص\n188 [cite: 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327]"
    },
    {
        "title": "الصفحة رقم 25",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\n4. عدم الإلمام الكامل بطبيعة التفويض في السلطات الإدارية، وعدم إدراك المزايا المهمة التي تعود على المنظمة الإدارية من استخدامه.\nويرجع ذلك إلى ضعف ثقافة الرئيس الإداري وضحالة معلوماته التي يجب أن يكون محيطاً بها.\nالفرع الثاني الصعوبات التي ترجع إلى المرؤوسين\nقد يكون الرؤساء راغبين في تفويض سلطاتهم، ومقتنعين بضرورة التفويض، ولكن المرؤوسين الذين يتم إليهم التفويض يترددون هم أنفسهم في قبول السلطة التي آلت إليهم بالتفويض، ويخشون من ممارستها.\nويعود تردد هؤلاء المرؤوسين إلى جملة من الأسباب أهمها :\n1. عدم ثقة هؤلاء المرؤوسين في أنفسهم، والخوف من الفشل في تحمل المسؤوليات والسلطات الجديدة.\n2 خوف المرؤوسين من فقد صداقاتهم القديمة؛ لأنهم يشعرون أنهم سوف يتميزون عن زملائهم بناء على مركزهم الجديد، والسلطات الممنوحة لهم، وهذا يباعد بينهم وبين زملائهم القدامي.\n3 خوف المرؤوسين من تصيّد الرؤساء الأخطائهم؛ لنقص ثقتهم في هؤلاء الرؤساء.\nانشير هنا إلى بعض الصعوبات العملية التي تعرقل عملية التفويض الإداري والتي لا يمكن أن نردها إلى الرؤساء أو المرؤوسين، وإنما إلى عوامل تنظيمية خاصة بالمنظمة الإدارية وتتمثل في عدم تحديد اختصاصات كل موظف في المنظمة بشكل دقيق نظراً لعدم وجود توصيف دقيق للوظائف، وكذلك عدم الاستقرار أو الثبات الوظيفي، ذلك أن كثرة التنقلات والندب والإعارات قد تدفع الرؤساء الإداريين إلى الإحجام عن التفويض.\nوأخيراً فإن صغر حجم المنظمة الإدارية وتمركزها في نطاق مكاني واحد يجعل تفويض الاختصاص قليل الأهمية.\n189 [cite: 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338]"
    },
    {
        "title": "الصفحة رقم 26",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nالمطلب الثالث: علاج الصعوبات التي تعترض التفويض الإداري\nذكرنا في المطلب السابق أهم المعوقات والصعوبات العملية التي تعترض عملية التفويض الإداري، وتؤثر في فعاليته في نطاق عمل المنظمة.\nوهذه الصعوبات الواقعية تفترض معالجة من خلال طرح مجموعة من الحلول لمواجهة هذه الصعوبات، وهذا ما سنقوم بتوضيحه في الفرعين الآتيين:\nالفرع الأول: علاج الصعوبات التي ترجع إلى الرؤساء الإداريين\n1. وجوب العمل على تغيير معتقدات الرؤساء وسلوكهم، ونظرتهم الخاطئة في كثير من الأحيان إلى التفويض، ويجب كذلك توعية هؤلاء الرؤساء وتعميق ثقافتهم، وإيضاح مزايا التفويض الإداري، من حيث كونه وسيلة فعالة لسير العمل الإداري بدقة وانتظام وبأعلى جودة ممكنة، وأنه لا يمثل سلباً لبعض سلطاتهم، وأنه ليس دليلاً على عدم كفاءتهم.\nويمكن للبرامج التدريبية التي تقوم بها الأجهزة الإدارية والمعاهد المتخصصة في الإدارة العامة (كالمعهد الوطني للإدارة العامة في سورية أن تؤدي دوراً في هذا المجال من خلال توعية الرؤساء الإداريين إلى أن مهمتهم ليست إصدار القرارات في كل الشؤون الإدارية الكبيرة منها والصغيرة، ولكنها تكمن في الإشراف والتوجيه ورسم السياسة العامة للمنظمة والتخطيط للمستقبل، وإعادة التنظيم للتغلب على المشكلات التي تواجه تحقيق المنظمة لأهدافها .\n2. العمل على تغيير نظرة الرؤساء إلى مرؤوسيهم ومدى كفاءتهم، وقدرتهم على مباشرة ما يوكل إليهم من مسؤوليات، ويجب أن يعرف الرؤساء أن عدم اشتراك أصحاب الكفاءة في مساعدته على إدارة المنظمة بعد خسارة كبيرة للمنظمة وله شخصياً، وأن هذا قد يدفعهم إلى ترك العمل بالمنظمة، والبحث عن مواقع للعمل تتفق مع طموحهم وقدراتهم.\n190 [cite: 339, 340, 341, 342, 343, 344, 345]"
    },
    {
        "title": "الصفحة رقم 27",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\n3. يجب أن تكون رقابة الرئيس على مرؤوسيه رقابة رشيدة وهادفة، إذ تكون غايتها تحقيق المصلحة العامة.\nويجب أن يكون لدى الرئيس المفوض استعداد لتقبل أخطاء المفوض إليهم بالاختصاص.\n4. يجب أن تراعى القيادة السياسية والإدارية عند اختيار الرؤساء الإداريين قدرتهم على ممارسة أعمال القيادة، وتوافر صفات القائد الإداري الكفء في كل رئيس منهم.\nومن أهم تلك الصفات القدرة على تفويض جانب من سلطاتهم إلى مرؤوسيهم حتى يتفرغوا للمهمات الرئيسة في إدارة المنظمة الإدارية.\nالفرع الثاني: علاج الصعوبات التي ترجع إلى المرؤوسين\n1. تدريب الموظفين على كيفية إنجاز الأعمال الإدارية المفوضة إليهم بأبسط الطرق وأسرعها، بما يحقق أحسن النتائج وأفضلها؛\nلأن نجاح عملية التفويض يعتمد إلى حد كبير على تمرّس المرؤوسين، وتعودهم على تلقي الاختصاصات وانجازها على أحسن وجه.\n2. يجب أن يكون المفوض إليه قادراً على أداء الواجبات، وتحمل المسؤوليات؛\nلذلك يجب على الرئيس الإداري عند قيامه بالتفويض أن يضع بحسبانه مدى إمكانيات المفوض إليه، وخبراته ومهاراته الشخصية، كي يضمن نجاحه في أداء المهمات المكلف بها.\n3 قيام الرؤساء الإداريين بتشجيع مرؤوسيهم، وإزالة مخاوفهم من عواقب المسؤولية عند القيام بمباشرة السلطات المفوضة إليهم، ويمكن للرئيس الإداري استخدام نظام الحوافز المادية والأدبية، للوصول إلى هذا الهدف.\nالخاتمة في ختام هذا البحث توصلت الدراسة إلى مجموعة من النتائج والمقترحات يمكن إجمالها على النحو الآتي:\nأولاً - النتائج:\n1. يعد التفويض الإداري أسلوباً مهماً للحد من التركيز الإداري، ولاسيما في وقتنا الحالي نظراً لتزايد وظائف الدولة، واتساع حجم سلطتها الإدارية.\n191 [cite: 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361]"
    },
    {
        "title": "الصفحة رقم 28",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\n2 حرصت الجمهورية العربية السورية على تقنين مبدأ التفويض الإداري، وتنظيم أحكامه من خلال القوانين المختلفة، كقانون الإدارة المحلية لعام 2011، وقانون تنظيم الجامعات رقم /6/ لعام 2006، ومن خلال إصدار تشريع عام ينظم عملية التفويض الإداري في الدولة هو المرسوم التشريعي رقم /69/ لعام 2005.\n3. مع أهمية التفويض الإداري ودوره في نطاق عمل المنظمة الإدارية، إلا أن هناك سلطات معينة مؤثرة في سياسة المنظمة ينبغي للرئيس الإداري الاحتفاظ بها وعدم تفويضها .\n4. يأخذ التفويض الإداري عادة صورتين : تفويض السلطة أو الاختصاص)، وتفويض التوقيع.\nوقد رتب الفقه على التمييز بين هاتين الصورتين نتائج قانونية مهمة.\n5. يخضع التفويض الإداري لعدد من المبادئ والشروط القانونية التي يمكن عدها موجهات ومحددات العملية التفويض.\n6. هنالك بعض النظم القانونية تتشابه مع نظام التفويض الإداري، كالحلول، والإنابة، ولكن كلا منها يتمتع بخصائص ذاتية تميزه من التفويض الإداري.\n7. يتسم التفويض الإداري بجملة من المزايا، ويحقق مجموعة من الفوائد تؤدي إلى أداء العمل الإداري في الوقت الأقصر والجودة الأعلى.\n8. تواجه عملية التفويض الإداري جملة من الصعوبات العملية، بعضها يتعلق بالرؤساء الإداريين، وبعضها الآخر يعود إلى المرؤوسين، وهذا الأمر يحتم العمل لمعالجة هذه الصعوبات.\nثانياً - المقترحات :\n1. يجب زيادة الوعي بأهمية إنجاز أهداف المنظمة، وذلك بزيادة مشاعر الإخلاص والتفاني لدى الرؤساء والمرؤوسين، وخلق روح العمل كفريق متكامل يعمل بنظام وبأسلوب متناسق للوصول إلى أهداف المنظمة.\n192 [cite: 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375]"
    },
    {
        "title": "الصفحة رقم 29",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\n2. تطبيق مبادئ التنظيم والإدارة العامة السليمة على المنظمات والإدارات الحكومية المختلفة، حتى تترسخ في الأذهان المزايا التي يحققها التفويض الإداري في إنجاز أهداف المنظمة، ومن ثم المساهمة في تنفيذ أهداف السياسة العامة للدولة.\n3 إيجاد خطوط اتصال سهلة وسريعة بين المنظمة وفروعها؛ لأن أعمال المنظمة متكاملة وحدوث خلل أو قصور في نشاط فرع معين سيؤثر سلباً في عمل بقية الفروع، وهذا سيؤثر في سير النشاط العام للمنظمة.\n4. ضرورة إقامة دورات تدريبية للرؤساء والمرؤوسين تبين مفهوم التفويض، وأنواعه، ومزاياه ودوره في تطوير العمل الإداري.\n5. إعادة النظر في الهياكل التنظيمية للوزارات، والأجهزة الحكومية، وتصنيف الوظائف العامة وتوصيفها، بما يسمح بإجراء التفويض الإداري بيسر وسهولة.\n6. ضرورة قيام المفوّض بتحديد مسؤولية المفوض إليه بشكل مكتوب ودقيق ضمن قرار التفويض درءاً للغموض، وتحديداً للمسؤوليات.\n193 [cite: 376, 377, 378, 379, 380, 381, 382]"
    },
    {
        "title": "الصفحة رقم 30",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nد. عمار التركاوي\nالمراجع\nأولاً - الكتب :\n1. د. ابرهيم عبد العزيز شيحا، أصول الإدارة العامة، منشأة المعارف، الإسكندرية عام 1993.\n2. د. حسين عثمان محمد عثمان أصول القانون الإداري، منشورات الحلبي الحقوقية بيروت عام 2008\n3. د. رمضان بطيخ ود. نوفان العجارمة، مبادئ القانون الإداري في المملكة الأردنية الهاشمية، ط1، دار إثراء للنشر والتوزيع، عمان، الأردن، عام 2011.\n4. د. سامي جمال الدين الوسيط في دعوى إلغاء القرارات الإدارية، ط1، منشأة المعارف، الإسكندرية، عام 2004.\n5. د. سعيد نحيلي القانون الإداري المبادئ العامة، ج1، منشورات جامعة البعث، عام .2013-2012\n6. د. سعيد نحيلي ود. عمار التركاوي، القانون الإداري المبادئ العامة\"، منشورات جامعة 2019-2018 دمشق عام\n7. د. طارق المجذوب، الإدارة العامة العملية الإدارية والوظيفة العامة والإصلاح الإداري ط1، منشورات الحلبي الحقوقية، بيروت، عام 2005.\n8. د. عبد الفتاح حسن، مبادئ الإدارة العامة، القاهرة عام 1972.\n9. د. عبد الغني بسيوني عبد الله أصول علم الإدارة العامة، الدار الجامعية، الإسكندرية عام 1992\n10. د. عبد الله طلبه الإدارة العامة، 8 ، منشورات جامعة دمشق عام 2007-2008\n11. د. عبد الله طلبه، مبادئ القانون الإداري، ج 1، منشورات جامعة دمشق، عام 2010-2011.\n12. د. عبد الله طلبه القانون الإداري الرقابة القضائية على أعمال الإدارة\"، منشورات 194\n2017-2016 جامعة دمشق، عام [cite: 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402]"
    },
    {
        "title": "الصفحة رقم 31",
        "abstract": "مجلة جامعة دمشق للعلوم الاقتصادية والقانونية - المجلد 37 - العدد الأول 2021\n13. د. عبد الله طلبه ود. محمد الحسين ود. مهند نوح المدخل إلى القانون الإداري منشورات جامعة دمشق نظام التعليم المفتوح، عام 2012-2013\n14. د. عيد قريطم التفويض في الاختصاصات الإدارية دراسة مقارنة، ط1، منشورات الحلبي الحقوقية، بيروت، عام 2011.\n15. د. ماجد راغب الحلو ، القانون الإداري، دار الجامعة الجديدة، الإسكندرية، عام 2006.\n16. د. ماجد راغب الحلو ، علم الإدارة العامة ومبادئ الشريعة الإسلامية، دار الجامعة الجديدة، الإسكندرية، عام 2007.\n17. د. محمد باهي أبو يونس، الضوابط الدستورية للوظيفة اللائحية التنفيذية دراسة مقارنة\"، دار الجامعة الجديدة، الإسكندرية، عام 2008.\n18. د. محمد الحسين ود. مهند نوح القانون الإداري عمال الإدارة العامة وتصرفاتها القانونية\"، منشورات جامعة دمشق عام 2011-2012.\n19. د. محمد رفعت عبد الوهاب، الإدارة العامة، دار الجامعة الجديدة، الإسكندرية، عام .2008-2007\n20. د. محمود عساف أصول الإدارة، مكتبة لطفي، القاهرة عام 1988.\n21. د. مصطفى أبو زيد فهمي، القضاء الإداري ومجلس الدولة، القاهرة، عام 1979.\n22. د. نجم الأحمد ود. أحمد اسماعيل الإدارة العامة، منشورات جامعة دمشق، عام 195\n.2017-2016 [cite: 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416]"
    },
    {
        "title": "الصفحة رقم 32",
        "abstract": "نظرية التفويض الإداري وإشكالاتها التطبيقية\nثانياً - الرسائل العلمية\nد. عمار التركاوي\n1. شادي اسماعيل اللوائح التنفيذية والرقابة القضائية عليها دراسة مقارنة\"، رسالة ماجستير في القانون العام، جامعة دمشق عام 2013.\n2. د. عبد الكريم الأحمد المصطفى، المفهوم القانوني لمبدأ الإدارة بالمشاركة في نطاق علم الإدارة العامة، أطروحة دكتوراه في القانون العام، جامعة دمشق، عام 2012.\nثالثاً - الدساتير والتشريعات\n1 دستور الجمهورية العربية السورية لعام 2012.\n2 المرسوم التشريعي رقم // 69 / لعام 2005 الناظم لتفويض الاختصاص في سورية.\n3 المرسوم التشريعي رقم / 107 / لعام 2011 المتضمن قانون الإدارة المحلية السوري وتعديلاته.\n4 قانون تنظيم الجامعات في سورية رقم /6 / لعام 2006 وتعديلاته.\nتاريخ ورود البحث : 2018/09/20\nتاريخ الموافقة على نشر البحث : 2018/10/30\n196 [cite: 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430]"
    }
]

In [3]:
# Resolve weights and construct classifier (tokenizer loads here; weights lazy on first predict)
model_path = str(resolve_arabert_model_path(get_settings()))
print("Model path:", model_path)
classifier = AdvancedArabicClassifier(model_path=model_path, idle_timeout_seconds=10)



⏳ جاري إعداد الموديل للتحليل...
⏳ جاري تحميل أدوات المعالجة والتوكينيزر (تبقى في الذاكرة دائماً)...

⚡ جاري إيقاظ الموديل وتحميله إلى الذاكرة (RAM/VRAM)...
✅ الكلاس جاهز للعمل على: CPU (مهلة الخمول: 10 ثانية)

💤 الكلاس في حالة خمول، جاري تفريغ الموديل لتحرير الذاكرة...

💤 الكلاس في حالة خمول، جاري تفريغ الموديل لتحرير الذاكرة...

💤 الكلاس في حالة خمول، جاري تفريغ الموديل لتحرير الذاكرة...

💤 الكلاس في حالة خمول، جاري تفريغ الموديل لتحرير الذاكرة...

💤 الكلاس في حالة خمول، جاري تفريغ الموديل لتحرير الذاكرة...

💤 الكلاس في حالة خمول، جاري تفريغ الموديل لتحرير الذاكرة...

💤 الكلاس في حالة خمول، جاري تفريغ الموديل لتحرير الذاكرة...


In [10]:
# ==============================================================================
# --- Block D: Execution on the PDF Extracted Batch ---
# ==============================================================================



# 3. التأكد من وجود البيانات (القائمة التي استخرجناها في الخطوة السابقة)
if 'articles_batch' in locals() and len(articles_batch) > 0:
    print(f"\n🚀 جاري تحليل عدد {len(articles_batch)} صفحات من البحث باستخدام الموديل...")

    # 4. إرسال الدفعة بالكامل إلى دالة التحليل
    batch_report = classifier.analyze_article_batch(articles_batch)
    print(batch_report)

    # 5. طباعة التقرير النهائي الأنيق
    print("\n" + "="*50)
    print("📑 التقرير النهائي لتحليل البحث (نظرية التفويض الإداري)")
    print("="*50)

    print(f"✅ إجمالي المقاطع (الصفحات) التي تم تحليلها: {batch_report['total_articles_analyzed']}")
    print(f"🏆 الفئة المسيطرة على هذا البحث هي: {batch_report['overall_dominant_class']}")

    print("\n📊 متوسط نسبة الثقة لكل فئة عبر جميع صفحات البحث:")

    # طباعة الفئات مرتبة تنازلياً حسب نسبة الثقة
    for label, avg_conf in batch_report['average_confidence_per_class'].items():
        # لتنظيف المخرجات، نعرض فقط الفئات التي حصلت على ثقة أعلى من 1%
        if avg_conf > 1.0:
            print(f"  🔸 {label}: {avg_conf}%")

    print("\n" + "="*50)
    print("💡 لمحة عن توقعات بعض الصفحات الفردية:")
    # طباعة نتيجة أول صفحتين كعينة
    for i in range(min(2, len(batch_report['individual_results']))):
        res = batch_report['individual_results'][i]
        print(f" - الصفحة {i+1}: تم تصنيفها كـ ({res['final_prediction']}) بثقة {res['confidence']}%")

else:
    print("❌ خطأ: القائمة articles_batch غير موجودة أو فارغة. تأكد من تشغيل كود الاستخراج أولاً.")


🚀 جاري تحليل عدد 32 صفحات من البحث باستخدام الموديل...
📊 جاري تحليل 32 مقالات (Threshold = 0.0)...


100%|██████████| 32/32 [00:13<00:00,  2.32it/s]

{'total_articles_analyzed': 32, 'applied_threshold': 0.0, 'average_confidence_per_class': {'العلوم الاقتصادية والسياسية': 66.11, 'العلوم القانونية': 32.38, 'الآداب والعلوم الإنسانية': 0.3, 'العلوم التربوية والنفسية': 0.28, 'العلوم الطبية': 0.18, 'الدراسات التاريخية': 0.17, 'العلوم الأساسية': 0.16, 'العلوم الزراعية': 0.16, 'العلوم الهندسية': 0.14, 'غير محدد': 0.11}, 'overall_dominant_class': 'العلوم الاقتصادية والسياسية', 'individual_results': [{'final_prediction': 'العلوم الاقتصادية والسياسية', 'original_top': 'العلوم الاقتصادية والسياسية', 'confidence': 97.05, 'all_probabilities': {'العلوم الاقتصادية والسياسية': 97.05, 'العلوم القانونية': 2.06, 'الآداب والعلوم الإنسانية': 0.14, 'العلوم الطبية': 0.14, 'العلوم التربوية والنفسية': 0.13, 'العلوم الزراعية': 0.13, 'الدراسات التاريخية': 0.1, 'العلوم الأساسية': 0.09, 'غير محدد': 0.08, 'العلوم الهندسية': 0.06}}, {'final_prediction': 'العلوم الاقتصادية والسياسية', 'original_top': 'العلوم الاقتصادية والسياسية', 'confidence': 82.69, 'all_probabil